# RQ1 – Baseline Performance

**Research Question:** How effectively can baseline supervised learning models predict sales revenue on the Marketing and Product Performance Dataset?

**Dataset:** [Marketing and Product Performance Dataset on Kaggle](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

**Instructions:** Upload the dataset CSV/Excel to your Kaggle notebook input, then update `DATA_PATH` below.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

# ── Configuration ────────────────────────────────────────────────────────────
# Update this path to your dataset file
import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42
TEST_SIZE = 0.2

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load & Inspect Data ───────────────────────────────────────────────────────
# Try CSV first, then Excel
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
df.head()

Shape: (10000, 17)

Column dtypes:
Campaign_ID                           object
Product_ID                            object
Budget                               float64
Clicks                                 int64
Conversions                            int64
Revenue_Generated                    float64
ROI                                  float64
Customer_ID                           object
Subscription_Tier                     object
Subscription_Length                    int64
Flash_Sale_ID                         object
Discount_Level                         int64
Units_Sold                             int64
Bundle_ID                             object
Bundle_Price                         float64
Customer_Satisfaction_Post_Refund      int64
Common_Keywords                       object
dtype: object

Missing values:
Campaign_ID                          0
Product_ID                           0
Budget                               0
Clicks                               0
Conversions  

,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative
2,CMP_6SBOWN,PROD_4V8A08,22367.45,3546,265,32585.62,1.41,CUST_BR2GST,Basic,9,FLASH_J4PEON,51,160,BNDL_0HY0EF,462.87,4,Affordable
3,CMP_Q31QCU,PROD_A1Q6ZB,29957.54,2573,781,95740.12,3.32,CUST_6TBY6K,Premium,32,FLASH_1TOVXT,36,159,BNDL_AI09BC,334.16,1,Durable
4,CMP_AY0UTJ,PROD_F57N66,36277.19,818,79,81990.43,3.53,CUST_XASI45,Standard,29,FLASH_AOBHXL,20,52,BNDL_R03ITT,371.67,2,Affordable


In [3]:
# ── Identify Target Variable ──────────────────────────────────────────────────
# Heuristic: pick column most likely to represent sales/revenue
candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
if candidates:
    TARGET = candidates[0]
else:
    # Fallback: pick numeric column with highest variance
    TARGET = df.select_dtypes(include=np.number).var().idxmax()
print(f'Target variable selected: {TARGET}')
print(df[TARGET].describe())

Target variable selected: Revenue_Generated
count    10000.000000
mean     50038.627579
std      28545.702337
min       1002.080000
25%      25264.255000
50%      49513.815000
75%      74507.157500
max      99999.470000
Name: Revenue_Generated, dtype: float64


In [4]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
df_clean = df.copy()

# Drop columns with >50% missing
thresh = len(df_clean) * 0.5
df_clean = df_clean.dropna(thresh=thresh, axis=1)

# Separate features and target
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# Encode categoricals
cat_cols = X.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Impute and scale
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (8000, 16), Test: (2000, 16)


In [5]:
# ── Train Baseline Models ─────────────────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(random_state=RANDOM_STATE),
    'k-NN':              KNeighborsRegressor(n_neighbors=5)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)})
    print(f'{name:25s} | MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}')

results_df = pd.DataFrame(results)
results_df

Linear Regression         | MAE=24757.9882 | RMSE=28604.0102 | R2=-0.0034
Decision Tree             | MAE=33620.2133 | RMSE=41091.0903 | R2=-1.0707
k-NN                      | MAE=26119.6576 | RMSE=31039.2160 | R2=-0.1815


,Model,MAE,RMSE,R2
0,Linear Regression,24757.9882,28604.0102,-0.0034
1,Decision Tree,33620.2133,41091.0903,-1.0707
2,k-NN,26119.6576,31039.2160,-0.1815


In [6]:
# ── Save Table ────────────────────────────────────────────────────────────────
results_df.to_csv('tables/RQ1_baseline_performance.csv', index=False)
print('Table saved: tables/RQ1_baseline_performance.csv')

Table saved: tables/RQ1_baseline_performance.csv


In [7]:
# ── Figure: Grouped Bar Chart ─────────────────────────────────────────────────
metrics = ['MAE', 'RMSE', 'R2']
x = np.arange(len(results_df))
width = 0.25
colors = ['#2E4057', '#048A81', '#54C6EB']

fig, ax = plt.subplots(figsize=(10, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, edgecolor='white', linewidth=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(results_df['Model'], fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Figure 1. Baseline Model Performance Comparison\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures/RQ1_baseline_performance.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved: figures/RQ1_baseline_performance.pdf')

Figure saved: figures/RQ1_baseline_performance.pdf


## Summary

This notebook answers **RQ1** by training three baseline supervised learning models (Linear Regression, Decision Tree, k-NN) and comparing their MAE, RMSE, and R² scores. Results are saved to `tables/RQ1_baseline_performance.csv` and `figures/RQ1_baseline_performance.pdf`.